# ASR training on Colab
Runtime → GPU (T4). Thư mục `ASR/` (kèm `data/` đã chuẩn bị) nằm trên Drive tại `PROJECT`.
Checkpoint ghi thẳng vào `experiments/<ID>/checkpoints` trên Drive → mất kết nối chỉ cần chạy lại các cell, train tự resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/DPL/ASR'  # sửa nếu đặt chỗ khác
%cd {PROJECT}
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy data từ Drive sang ổ local của Colab (đọc nhanh hơn nhiều khi train)
!rsync -a --info=progress2 {PROJECT}/data/ /content/data/
DATA = '/content/data'

In [ ]:
# Baseline zero-shot (không train) trên DEV
!python scripts/evaluate.py B0_phowhisper_small_zeroshot --split dev --data-dir {DATA}
!python scripts/evaluate.py B1_whisper_small_zeroshot --split dev --data-dir {DATA}

In [ ]:
# Train (tự resume từ checkpoint mới nhất nếu có)
EXP = 'V1_phowhisper_small_lora'
!python scripts/train.py {EXP} --data-dir {DATA}

In [ ]:
# Đánh giá model tốt nhất (experiments/<EXP>/final) trên DEV đầy đủ
!python scripts/evaluate.py {EXP} --split dev --data-dir {DATA}